# 후보 모델에 대해 하이퍼파라미터 선택
### 후보 모델
1. 로지스틱 회귀 + 특성 v7 사용

In [1]:
# 데이터셋 로드
import pandas as pd

train_origin = pd.read_csv("../data/processed/01/train.csv")

## 후보 1 테스트


In [2]:
from src.feature import prep_v7
from src.feature import prep_v5
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

prep = prep_v7()
model = make_pipeline(prep, LogisticRegression())

파라미터 조합 설정

In [3]:
param_grid = [{
    "logisticregression__C": [0.001, 0.01, 0.1,0.5, 1, 10, 100],
    "logisticregression__l1_ratio": [1],
    "logisticregression__max_iter": [10000],
    "logisticregression__solver": ["liblinear"],
     "logisticregression__class_weight": [None, "balanced"]
},
    {
    "logisticregression__C": [0.001, 0.01, 0.1,0.5, 1, 10, 100],
    "logisticregression__l1_ratio": [0],
    "logisticregression__max_iter": [10000],
    "logisticregression__solver": ["lbfgs"],
         "logisticregression__class_weight": [None, "balanced"]
},
]

하이퍼 파라미터 탐색 객체 생성

In [4]:
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=10,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)


학습 및 결과확인


In [5]:
train = train_origin.copy()
labels = train["Survived"]

grid_search.fit(train, labels)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'logisticregression__C': 10, 'logisticregression__class_weight': None, 'logisticregression__l1_ratio': 1, 'logisticregression__max_iter': 10000, 'logisticregression__solver': 'liblinear'}
0.8189553990610328


In [6]:

results = grid_search.cv_results_
#딕셔너리 형태라 데이터프레임으로 바꿔보는게 좋음
results_df = pd.DataFrame(results).sort_values("rank_test_score")
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_logisticregression__C,param_logisticregression__class_weight,param_logisticregression__l1_ratio,param_logisticregression__max_iter,param_logisticregression__solver,params,...,split2_train_score,split3_train_score,split4_train_score,split5_train_score,split6_train_score,split7_train_score,split8_train_score,split9_train_score,mean_train_score,std_train_score
26,0.018943,0.010035,0.005368,0.000896,100.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 100, 'logisticregres...",...,0.814353,0.819033,0.812793,0.826833,0.825273,0.833073,0.826833,0.825273,0.824909,0.007049
12,0.028870,0.006947,0.007214,0.001799,100.000,NaN,1,10000,liblinear,"{'logisticregression__C': 100, 'logisticregres...",...,0.814353,0.819033,0.812793,0.826833,0.825273,0.833073,0.826833,0.825273,0.824909,0.007049
10,0.017027,0.003735,0.005294,0.001028,10.000,NaN,1,10000,liblinear,"{'logisticregression__C': 10, 'logisticregress...",...,0.814353,0.823713,0.814353,0.826833,0.823713,0.833073,0.826833,0.826833,0.825533,0.006553
20,0.018226,0.007715,0.007231,0.002586,0.500,NaN,0,10000,lbfgs,"{'logisticregression__C': 0.5, 'logisticregres...",...,0.817473,0.822153,0.819033,0.828393,0.829953,0.826833,0.825273,0.829953,0.826781,0.005727
22,0.015083,0.004510,0.013504,0.012435,1.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 1, 'logisticregressi...",...,0.814353,0.822153,0.822153,0.828393,0.833073,0.826833,0.826833,0.831513,0.827249,0.006029
24,0.022988,0.007983,0.008233,0.003739,10.000,NaN,0,10000,lbfgs,"{'logisticregression__C': 10, 'logisticregress...",...,0.814353,0.822153,0.819033,0.826833,0.822153,0.833073,0.828393,0.826833,0.825845,0.006070
21,0.014960,0.004738,0.007982,0.003242,0.500,balanced,0,10000,lbfgs,"{'logisticregression__C': 0.5, 'logisticregres...",...,0.806552,0.803432,0.808112,0.815913,0.808112,0.822153,0.815913,0.819033,0.813516,0.006050
16,0.015118,0.004509,0.007094,0.002846,0.010,NaN,0,10000,lbfgs,"{'logisticregression__C': 0.01, 'logisticregre...",...,0.806552,0.815913,0.809672,0.803432,0.814353,0.820593,0.817473,0.815913,0.812421,0.005143
8,0.011582,0.002059,0.009768,0.010890,1.000,NaN,1,10000,liblinear,"{'logisticregression__C': 1, 'logisticregressi...",...,0.814353,0.822153,0.825273,0.828393,0.828393,0.834633,0.823713,0.831513,0.827249,0.006002
18,0.017641,0.004102,0.006717,0.001831,0.100,NaN,0,10000,lbfgs,"{'logisticregression__C': 0.1, 'logisticregres...",...,0.815913,0.812793,0.820593,0.817473,0.809672,0.823713,0.814353,0.829953,0.819134,0.005898


### 최적 모델 저장

In [7]:
import joblib
best_model = grid_search.best_estimator_
joblib.dump(best_model, "../data/model/tuned_logistic_regression.pkl")

['../data/model/tuned_logistic_regression.pkl']